**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 31: 온톨로지 RAG — RDF · OWL 추론 (rdflib)

## 학습 목표

1️⃣ **온톨로지 = 사실(ABox) + 스키마·규칙(TBox)** 임을 이해한다
2️⃣ **그래프 RAG vs 온톨로지 RAG**의 핵심 차이가 **추론(reasoning)** 임을 안다
3️⃣ `rdflib`로 **RDF 트리플 + RDFS/OWL 스키마**를 구축한다
4️⃣ **SPARQL**로 질의하고, **OWL/RDFS 추론**으로 *명시하지 않은 사실*을 도출한다
5️⃣ **제약(거버넌스 규칙) 검사**와 **온톨로지 RAG + LLM** 연동을 실습한다
6️⃣ 실서비스용 **온톨로지 DB 선택**(Fuseki / GraphDB / Ontop)을 익힌다

---

### ⚠️ 30강(그래프 RAG)과의 관계
- **30강**: Neo4j/NetworkX로 **저장된 관계를 탐색**(traversal) → *그래프 RAG*
- **31강**: 그 위에 **스키마+규칙**을 얹어 *저장 안 한 사실까지 추론* → *온톨로지 RAG*
- 프로퍼티 그래프(Neo4j·Apache AGE)는 OWL 추론이 없어 **그래프 RAG 도구**입니다.
  온톨로지 RAG는 **RDF/OWL 스택**(rdflib → Fuseki/GraphDB)을 씁니다.

### 실습 환경
- **필수 패키지**: `rdflib`, `openai`(선택), `python-dotenv`
- **외부 서비스 불필요** (rdflib 메모리 그래프로 전 과정 실습)

In [ ]:
# 패키지 임포트 및 환경 설정
import os
import json
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL
from urllib.parse import quote
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # OPENAI_API_KEY 등 (있으면 §8에서 사용)

print("rdflib 준비 완료")
print("OPENAI_API_KEY:", "✅" if os.getenv("OPENAI_API_KEY") else "❌ (없으면 §8 LLM 셀은 건너뜀)")

---
## 1️⃣ 그래프 RAG vs 온톨로지 RAG — 핵심은 "추론"

| | 그래프 RAG (30강) | **온톨로지 RAG (31강)** |
|---|---|---|
| 저장 | 사실(노드·엣지) | 사실 + **스키마·규칙(TBox)** |
| 질의 | 저장된 엣지 **탐색** | 저장 안 된 사실도 **추론** |
| DB | 프로퍼티 그래프 (Neo4j·AGE) | **RDF 트리플스토어** (Fuseki·GraphDB) |
| 질의어 | Cypher | **SPARQL** |
| 답하는 것 | "그래프에 **있는** 것" | "그래프에서 **논리적으로 따라 나오는** 것" |

**온톨로지가 추가하는 것 = 추론.** 예:
- `회장 ⊑ 사람` → 회장만 저장해도 "사람을 보여줘"에 포함 (클래스 계층)
- `경쟁 = 대칭` → 한 방향만 저장해도 양방향 도출 (속성 특성)
- `1인 1회장` → 규칙 위반 자동 감지 (제약)

---
## 2️⃣ RDF 기초 — 모든 지식을 트리플로

**RDF**는 지식을 `(주어, 술어, 목적어)` **트리플**로 표현합니다.

```
(이재용) --[chairmanOf]--> (삼성그룹)
 주어        술어            목적어
```

- **TBox** (스키마/용어): 클래스·관계의 정의와 규칙 — "회장은 사람의 한 종류"
- **ABox** (사실/단언): 실제 인스턴스 — "이재용은 회장이다"

아래 작은 예제로 트리플이 어떻게 생겼는지 봅니다.

In [ ]:
# RDF 첫걸음 — 트리플 3개
EX = Namespace("http://example.org/")
demo = Graph(); demo.bind("ex", EX)

demo.add((EX["이재용"], RDF.type,   EX.Chairman))        # 이재용 은 회장(Chairman) 이다
demo.add((EX["이재용"], RDFS.label, Literal("이재용")))   # 사람이 읽는 이름
demo.add((EX["이재용"], EX.chairmanOf, EX["삼성그룹"]))   # 이재용 --chairmanOf--> 삼성그룹

print(f"트리플 {len(demo)}개 — Turtle 직렬화:")
print(demo.serialize(format="turtle"))

---
## 3️⃣ 도메인 데이터 — 한국 대기업 거버넌스

그룹·계열사·회장·CEO·학교, 그리고 산업/경쟁사 관계를 사실(ABox)로 준비합니다.
(30강과 같은 도메인 — "같은 데이터, 추론을 더했을 때 무엇이 달라지나"를 보기 위함)

In [ ]:
corporate_data = {
    "nodes": [
        {"label": "Group", "props": {"name": "삼성그룹",       "founded_year": 1938}},
        {"label": "Group", "props": {"name": "SK그룹",         "founded_year": 1953}},
        {"label": "Group", "props": {"name": "현대자동차그룹", "founded_year": 1947}},
        {"label": "Person", "props": {"name": "이재용", "role": "회장"}},
        {"label": "Person", "props": {"name": "최태원", "role": "회장"}},
        {"label": "Person", "props": {"name": "정의선", "role": "회장"}},
        {"label": "Subsidiary", "props": {"name": "삼성전자"}},
        {"label": "Subsidiary", "props": {"name": "삼성물산"}},
        {"label": "Subsidiary", "props": {"name": "삼성SDS"}},
        {"label": "Subsidiary", "props": {"name": "SK하이닉스"}},
        {"label": "Subsidiary", "props": {"name": "SK이노베이션"}},
        {"label": "Subsidiary", "props": {"name": "SK텔레콤"}},
        {"label": "Subsidiary", "props": {"name": "현대자동차"}},
        {"label": "Subsidiary", "props": {"name": "기아"}},
        {"label": "Subsidiary", "props": {"name": "현대모비스"}},
        {"label": "Person", "props": {"name": "한종희", "role": "CEO"}},
        {"label": "Person", "props": {"name": "오세철", "role": "CEO"}},
        {"label": "Person", "props": {"name": "이준희", "role": "CEO"}},
        {"label": "Person", "props": {"name": "곽노정", "role": "CEO"}},
        {"label": "Person", "props": {"name": "박상규", "role": "CEO"}},
        {"label": "Person", "props": {"name": "유영상", "role": "CEO"}},
        {"label": "Person", "props": {"name": "장재훈", "role": "CEO"}},
        {"label": "Person", "props": {"name": "송호성", "role": "CEO"}},
        {"label": "Person", "props": {"name": "이규석", "role": "CEO"}},
    ],
    "edges": [
        {"from": "삼성그룹", "rel": "HAS_CHAIRMAN", "to": "이재용"},
        {"from": "SK그룹", "rel": "HAS_CHAIRMAN", "to": "최태원"},
        {"from": "현대자동차그룹", "rel": "HAS_CHAIRMAN", "to": "정의선"},
        {"from": "삼성전자", "rel": "SUBSIDIARY_OF", "to": "삼성그룹"},
        {"from": "삼성물산", "rel": "SUBSIDIARY_OF", "to": "삼성그룹"},
        {"from": "삼성SDS", "rel": "SUBSIDIARY_OF", "to": "삼성그룹"},
        {"from": "SK하이닉스", "rel": "SUBSIDIARY_OF", "to": "SK그룹"},
        {"from": "SK이노베이션", "rel": "SUBSIDIARY_OF", "to": "SK그룹"},
        {"from": "SK텔레콤", "rel": "SUBSIDIARY_OF", "to": "SK그룹"},
        {"from": "현대자동차", "rel": "SUBSIDIARY_OF", "to": "현대자동차그룹"},
        {"from": "기아", "rel": "SUBSIDIARY_OF", "to": "현대자동차그룹"},
        {"from": "현대모비스", "rel": "SUBSIDIARY_OF", "to": "현대자동차그룹"},
        {"from": "삼성전자", "rel": "HAS_CEO", "to": "한종희"},
        {"from": "삼성물산", "rel": "HAS_CEO", "to": "오세철"},
        {"from": "삼성SDS", "rel": "HAS_CEO", "to": "이준희"},
        {"from": "SK하이닉스", "rel": "HAS_CEO", "to": "곽노정"},
        {"from": "SK이노베이션", "rel": "HAS_CEO", "to": "박상규"},
        {"from": "SK텔레콤", "rel": "HAS_CEO", "to": "유영상"},
        {"from": "현대자동차", "rel": "HAS_CEO", "to": "장재훈"},
        {"from": "기아", "rel": "HAS_CEO", "to": "송호성"},
        {"from": "현대모비스", "rel": "HAS_CEO", "to": "이규석"},
        {"from": "이재용", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "최태원", "rel": "GRADUATED_FROM", "to": "시카고대학교"},
        {"from": "정의선", "rel": "GRADUATED_FROM", "to": "고려대학교"},
        {"from": "한종희", "rel": "GRADUATED_FROM", "to": "인하대학교"},
        {"from": "오세철", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "이준희", "rel": "GRADUATED_FROM", "to": "연세대학교"},
        {"from": "곽노정", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "박상규", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "유영상", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "장재훈", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "송호성", "rel": "GRADUATED_FROM", "to": "한양대학교"},
        {"from": "이규석", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        # 경쟁 관계 — '한 방향만' 저장 (대칭 추론으로 반대 방향을 도출할 것)
        {"from": "SK하이닉스", "rel": "COMPETES_WITH", "to": "삼성전자"},
    ],
}
print(f"노드 {len(corporate_data['nodes'])}개, 엣지 {len(corporate_data['edges'])}개 준비")

---
## 4️⃣ rdflib 로 온톨로지 구축 — TBox(스키마) + ABox(사실)

- **TBox**: 클래스 계층(`rdfs:subClassOf`), 속성 특성(`owl:SymmetricProperty`, `owl:inverseOf`)
- **ABox**: `corporate_data` 의 사실을 트리플로 변환

In [ ]:
GOV = Namespace("http://example.org/gov#")
g = Graph(); g.bind("gov", GOV); g.bind("owl", OWL)
def uri(name): return GOV[quote(name)]

# ── TBox: 스키마와 규칙 ──
for c in ["Group", "Subsidiary", "Person", "University", "Agent"]:
    g.add((GOV[c], RDF.type, OWL.Class))
g.add((GOV.Chairman, RDFS.subClassOf, GOV.Person))   # 회장 ⊑ 사람
g.add((GOV.CEO,      RDFS.subClassOf, GOV.Person))   # CEO  ⊑ 사람
g.add((GOV.Person,   RDFS.subClassOf, GOV.Agent))    # 사람 ⊑ 행위자  (계층 전이 시연)
g.add((GOV.competesWith, RDF.type, OWL.SymmetricProperty))  # 경쟁 = 대칭
g.add((GOV.hasCEO, OWL.inverseOf, GOV.ceoOf))               # hasCEO 의 역관계 = ceoOf

REL = {"HAS_CHAIRMAN": GOV.hasChairman, "SUBSIDIARY_OF": GOV.subsidiaryOf,
       "HAS_CEO": GOV.hasCEO, "GRADUATED_FROM": GOV.graduatedFrom,
       "COMPETES_WITH": GOV.competesWith}
LABEL = {"Group": GOV.Group, "Subsidiary": GOV.Subsidiary, "Person": GOV.Person}

# ── ABox: 사실 ──
known = set()
for node in corporate_data["nodes"]:
    nm = node["props"]["name"]; known.add(nm)
    role = node["props"].get("role"); cls = LABEL.get(node["label"], OWL.Thing)
    if node["label"] == "Person" and role == "회장": cls = GOV.Chairman
    elif node["label"] == "Person" and role == "CEO": cls = GOV.CEO
    g.add((uri(nm), RDF.type, cls)); g.add((uri(nm), RDFS.label, Literal(nm)))

for e in corporate_data["edges"]:
    for ep in (e["from"], e["to"]):
        if ep not in known:
            g.add((uri(ep), RDFS.label, Literal(ep))); known.add(ep)
    if e["rel"] == "GRADUATED_FROM":
        g.add((uri(e["to"]), RDF.type, GOV.University))
    g.add((uri(e["from"]), REL[e["rel"]], uri(e["to"])))

print(f"RDF 트리플 {len(g)}개 구축\n")
print("Turtle 미리보기 (일부):")
print("\n".join(g.serialize(format="turtle").splitlines()[3:13]))

---
### 🛠️ 실습 도우미 — 결과를 **표**·**그림**으로 보기

SPARQL 결과를 pandas **표**(DataFrame)로, 온톨로지를 networkx **그래프 그림**으로 봅니다.
(설치: `pip install pandas matplotlib networkx`)

In [ ]:
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import networkx as nx
matplotlib.rcParams["font.family"] = "NanumGothic"      # 한글 라벨
matplotlib.rcParams["axes.unicode_minus"] = False

def sparql_df(query, graph=g):
    """SPARQL 결과 → pandas DataFrame (셀 마지막 줄에 두면 표로 렌더)."""
    res = graph.query(query)
    return pd.DataFrame([{str(v): str(r[v]) for v in res.vars} for r in res])

_REL_KO = {GOV.hasChairman:"회장", GOV.subsidiaryOf:"계열사", GOV.hasCEO:"CEO",
           GOV.graduatedFrom:"졸업", GOV.competesWith:"경쟁"}

def draw_ontology(graph, title="온톨로지 그래프"):
    """인스턴스 관계를 그래프로 시각화 (rdfs:label 사용)."""
    lbl = {s:str(o) for s,_,o in graph.triples((None, RDFS.label, None))}
    G = nx.DiGraph()
    for prop, ko in _REL_KO.items():
        for s,_,o in graph.triples((None, prop, None)):
            if s in lbl and o in lbl: G.add_edge(lbl[s], lbl[o], label=ko)
    plt.figure(figsize=(13,9))
    pos = nx.spring_layout(G, k=0.9, seed=42)
    nx.draw(G, pos, with_labels=True, node_color="#9ec5fe", node_size=1400,
            font_size=9, font_family="NanumGothic", edge_color="gray", arrows=True, arrowsize=15)
    nx.draw_networkx_edge_labels(G, pos, font_family="NanumGothic", font_size=7,
            edge_labels=nx.get_edge_attributes(G, "label"))
    plt.title(title, fontsize=14); plt.axis("off"); plt.tight_layout(); plt.show()

print("헬퍼 준비 완료: sparql_df(query) / draw_ontology(graph)")

In [ ]:
# 온톨로지 그래프 시각화 (인스턴스 관계)
draw_ontology(g, title="기업 거버넌스 온톨로지")

---
## 5️⃣ SPARQL 질의

SPARQL 은 그래프용 질의어입니다. 멀티홉 탐색 자체는 **그래프 RAG(30강)와 공유**하는 능력이며, 31강의 차별점은 **다음 절의 추론**입니다.

In [ ]:
PFX = """PREFIX gov: <http://example.org/gov#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>"""

print("[Q1] 모든 그룹")
for r in g.query(PFX + " SELECT ?n WHERE { ?x a gov:Group ; rdfs:label ?n }"):
    print("  -", str(r.n))

print("\n[Q2] 4-hop: 이재용이 회장인 그룹의 계열사 CEO 들의 학교 (그래프 RAG 와 공유)")
q2 = PFX + """
SELECT ?ceo ?school WHERE {
  ?grp gov:hasChairman ?c .  ?c rdfs:label "이재용" .
  ?sub gov:subsidiaryOf ?grp ; gov:hasCEO ?p .
  ?p rdfs:label ?ceo ; gov:graduatedFrom ?u .  ?u rdfs:label ?school .
} ORDER BY ?school"""
for r in g.query(q2):
    print(f"   {r.ceo} → {r.school}")

In [ ]:
# 같은 4-hop 결과를 '표'로 보기 (sparql_df 헬퍼)
df = sparql_df(q2)
print(df.to_string(index=False))
df  # Jupyter 에서 HTML 표로 렌더

---
## 6️⃣ OWL / RDFS 추론 — ★ 온톨로지 RAG의 핵심

**명시하지 않은 사실**을 규칙으로 자동 도출합니다. 그래프 RAG(30강)는 못 하는 부분.

적용 규칙(전방향 추론):
- **RDFS**: `(s a C) ∧ (C ⊑ D) ⟹ (s a D)`  / `subClassOf` 전이
- **OWL**: `SymmetricProperty`(대칭), `inverseOf`(역관계), `TransitiveProperty`(전이)

In [ ]:
def reason(graph):
    """간단한 전방향 추론기 (RDFS subClassOf + OWL Symmetric/Inverse/Transitive)."""
    base = len(graph); changed = True
    while changed:
        changed = False; new = []
        # RDFS: 타입 전파 + subClassOf 전이
        for s, _, c in graph.triples((None, RDF.type, None)):
            for _, _, d in graph.triples((c, RDFS.subClassOf, None)):
                new.append((s, RDF.type, d))
        for c, _, d in graph.triples((None, RDFS.subClassOf, None)):
            for _, _, e in graph.triples((d, RDFS.subClassOf, None)):
                new.append((c, RDFS.subClassOf, e))
        # OWL 대칭
        for p, _, _ in graph.triples((None, RDF.type, OWL.SymmetricProperty)):
            for x, _, y in graph.triples((None, p, None)):
                new.append((y, p, x))
        # OWL 역관계
        for p, _, q in graph.triples((None, OWL.inverseOf, None)):
            for x, _, y in graph.triples((None, p, None)):
                new.append((y, q, x))
        # OWL 전이
        for p, _, _ in graph.triples((None, RDF.type, OWL.TransitiveProperty)):
            for x, _, y in graph.triples((None, p, None)):
                for _, _, z in graph.triples((y, p, None)):
                    new.append((x, p, z))
        for t in new:
            if t not in graph:
                graph.add(t); changed = True
    return len(graph) - base

q_person = PFX + " SELECT ?n WHERE { ?p a gov:Person ; rdfs:label ?n }"
before = {str(r.n) for r in g.query(q_person)}
print(f"[추론 전] gov:Person 직접 인스턴스: {len(before)}명  (회장/CEO 는 Person 으로 안 잡힘)")

added = reason(g)
after = {str(r.n) for r in g.query(q_person)}
print(f"[추론] 새 트리플 {added}개 생성")
print(f"[추론 후] gov:Person: {len(after)}명 → subClassOf 로 추론: {sorted(after - before)[:5]} ...")

# 대칭 추론
print("\n[대칭] competesWith — 한 방향만 줬는데 양방향 도출:")
for r in g.query(PFX + " SELECT ?a ?b WHERE { ?x gov:competesWith ?y . ?x rdfs:label ?a . ?y rdfs:label ?b }"):
    print(f"   {r.a} <-> {r.b}")

# 역관계 추론
print("\n[역관계] hasCEO 의 역관계 ceoOf 자동 생성 (예시 3개):")
for r in list(g.query(PFX + " SELECT ?ceo ?comp WHERE { ?p gov:ceoOf ?c . ?p rdfs:label ?ceo . ?c rdfs:label ?comp }"))[:3]:
    print(f"   {r.ceo} 는 {r.comp} 의 CEO")

---
## 7️⃣ 제약 / 일관성 검사 — 거버넌스 규칙

온톨로지는 "있어야 할/있으면 안 될" 상태를 **규칙으로 검사**할 수 있습니다.
예: **1인 1회장** — 한 사람이 두 개 이상 그룹의 회장이면 위반.

In [ ]:
def check_one_chairman(graph):
    q = PFX + """
    SELECT ?person (COUNT(?grp) AS ?cnt) WHERE {
      ?grp gov:hasChairman ?p . ?p rdfs:label ?person
    } GROUP BY ?person HAVING (COUNT(?grp) > 1)"""
    return [(str(r.person), int(r.cnt)) for r in graph.query(q)]

print("[정상 상태] 1인 1회장 위반 검사:", check_one_chairman(g) or "위반 없음 ✅")

# 위반 시나리오: 이재용을 SK그룹 회장으로도 임명해 본다
g.add((uri("SK그룹"), GOV.hasChairman, uri("이재용")))
print("[위반 추가] 이재용을 SK그룹 회장으로도 임명 →", check_one_chairman(g))
g.remove((uri("SK그룹"), GOV.hasChairman, uri("이재용")))   # 원복
print("[원복 후]", check_one_chairman(g) or "위반 없음 ✅")

---
## 8️⃣ 온톨로지 RAG + LLM

**추론된 그래프**에서 SPARQL 로 사실을 검색(retrieve)하고, 그 사실을 LLM 에 넘겨 자연어 답을 생성합니다.
포인트: 검색 대상이 *추론으로 보강된* 그래프라, 그래프 RAG 가 놓치는 사실까지 답합니다.

In [ ]:
question = "삼성전자와 경쟁하는 회사는 어디인가요?"

# 1) 추론된 그래프에서 관련 사실 검색 (대칭 추론 덕에 양방향 모두 잡힘)
facts = [f"{r.a} competesWith {r.b}" for r in g.query(
    PFX + " SELECT ?a ?b WHERE { ?x gov:competesWith ?y . ?x rdfs:label ?a . ?y rdfs:label ?b }")]
retrieved = [f for f in facts if "삼성전자" in f]
print("검색된 사실:", retrieved)

# 2) LLM 으로 자연어 답변 (키 있을 때만)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    from openai import OpenAI
    client = OpenAI()
    prompt = f"다음 사실만 근거로 질문에 한국어로 간결히 답하세요.\n사실: {retrieved}\n질문: {question}"
    resp = client.chat.completions.create(model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}], temperature=0)
    print("\n[LLM 답변]", resp.choices[0].message.content)
else:
    print("\n(OPENAI_API_KEY 없음 — 검색된 사실로 답: 삼성전자의 경쟁사는 SK하이닉스)")

---
## 9️⃣ 그래프 RAG vs 온톨로지 RAG — 같은 질문, 다른 답

"삼성전자의 경쟁사는?" — 데이터에는 `SK하이닉스 →competesWith→ 삼성전자` **한 방향만** 저장돼 있습니다.

In [ ]:
mini = Graph(); mini.bind("gov", GOV)
mini.add((GOV.competesWith, RDF.type, OWL.SymmetricProperty))
mini.add((uri("SK하이닉스"), GOV.competesWith, uri("삼성전자")))  # 한 방향만
mini.add((uri("삼성전자"), RDFS.label, Literal("삼성전자")))
mini.add((uri("SK하이닉스"), RDFS.label, Literal("SK하이닉스")))

q = PFX + """ SELECT ?n WHERE {
  ?s rdfs:label "삼성전자" . ?s gov:competesWith ?x . ?x rdfs:label ?n }"""

print("질문: 삼성전자의 경쟁사는?")
print("  [그래프 RAG] 저장된 방향만 탐색 →", [str(r.n) for r in mini.query(q)] or "(없음)")
reason(mini)
print("  [온톨로지 RAG] 대칭 추론 후     →", [str(r.n) for r in mini.query(q)])
print("\n→ 같은 데이터·같은 질문인데, 추론이 있으면 답이 달라진다 (이것이 온톨로지 RAG)")

---
## 9️⃣.5 실제 오픈소스 RDF DB 로 SPARQL — **Oxigraph**

지금까지는 rdflib **메모리** 그래프였습니다. 실서비스처럼 **디스크에 영속되는
오픈소스 트리플스토어**에 적재해 SPARQL 로 질의합니다.

- **Oxigraph** = Rust 기반 임베디드 RDF DB (`pip install pyoxigraph`) — **Java·서버 불필요**
- 패턴: rdflib 에서 **추론까지 끝낸 트리플**을 DB 에 적재(materialize) → SPARQL 서비스
- **Jena Fuseki** 는 같은 일을 **HTTP SPARQL 엔드포인트**로 제공(Java 필요).
  GraphDB/Stardog 은 적재 시 OWL 추론을 DB 가 자동 수행.

In [ ]:
from pyoxigraph import Store, RdfFormat
import shutil

DB_PATH = "oxigraph_db"
shutil.rmtree(DB_PATH, ignore_errors=True)
store = Store(DB_PATH)                                   # 디스크 영속 트리플스토어
store.load(g.serialize(format="nt").encode("utf-8"),    # 추론까지 끝낸 rdflib 그래프를 적재
           format=RdfFormat.N_TRIPLES)
print(f"Oxigraph 적재 완료 — 트리플 {len(store)}개 (추론분 포함)\n")

PFX_S = """PREFIX gov: <http://example.org/gov#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>"""

print("[Oxigraph SPARQL] 이재용 그룹 계열사 CEO 들의 학교:")
q = PFX_S + """ SELECT ?ceo ?school WHERE {
  ?grp gov:hasChairman ?c . ?c rdfs:label "이재용" .
  ?sub gov:subsidiaryOf ?grp ; gov:hasCEO ?p .
  ?p rdfs:label ?ceo ; gov:graduatedFrom ?u . ?u rdfs:label ?school .
} ORDER BY ?school"""
for sol in store.query(q):
    print(f"   {sol['ceo'].value} → {sol['school'].value}")

print("\n[Oxigraph SPARQL] 삼성전자 경쟁사 (대칭 추론분도 DB 에 적재됨):")
for sol in store.query(PFX_S + """ SELECT ?n WHERE {
  ?s rdfs:label "삼성전자" . ?s gov:competesWith ?x . ?x rdfs:label ?n }"""):
    print("  -", sol["n"].value)

del store                                               # 닫기
store2 = Store(DB_PATH)                                  # 재오픈 → 영속 확인
print(f"\n재오픈 후 트리플 {len(store2)}개 — 디스크에 영속됨 ✅")
del store2
shutil.rmtree(DB_PATH, ignore_errors=True)              # 실습 후 정리

---
## 🔟 실서비스: 온톨로지 RAG 용 DB 선택

메모리 rdflib 는 실습/프로토타입용입니다. 프로덕션은 **OWL 추론을 지원하는 RDF 트리플스토어**가 필요합니다.

| 상황 | DB | 비고 |
|---|---|---|
| 표준/오픈소스 | **Apache Jena + Fuseki** | RDF·SPARQL·추론, SPARQL 엔드포인트 |
| 프로덕션 추론 | **GraphDB / Stardog** | OWL RL 추론 강력 |
| 기존 관계형 DB에 온톨로지 | **Ontop + PostgreSQL** | **OWL QL**(질의 재작성, OBDA) |
| 관리형 클라우드 | **Amazon Neptune** | RDF/SPARQL (추론은 제한적) |
| 실습/프로토타입 | **rdflib (+owlrl)** | 본 노트북 |

> ❌ **Neo4j · Apache AGE** = 프로퍼티 그래프 → OWL 추론 없음 → *그래프 RAG(30강)용*. 온톨로지 RAG 에는 부적합.

In [ ]:
# (1) 메모리 그래프를 파일로 영속화 — 표준 RDF 포맷(Turtle)
g.serialize(destination="corporate_ontology.ttl", format="turtle")
print("저장: corporate_ontology.ttl  (트리플", len(g), "개)")

# (2) 프로덕션 트리플스토어(Fuseki/GraphDB)에 올리고 SPARQL 엔드포인트로 질의 — 개념 코드
print("""
# ── Apache Jena Fuseki 예시 (개념) ─────────────────────────
#   docker run -p 3030:3030 stain/jena-fuseki        # Fuseki 기동
#   # 데이터셋 생성 후 corporate_ontology.ttl 업로드
#
#   from SPARQLWrapper import SPARQLWrapper, JSON
#   sparql = SPARQLWrapper("http://localhost:3030/corp/sparql")
#   sparql.setQuery(PFX + " SELECT ?n WHERE { ?x a gov:Group ; rdfs:label ?n }")
#   sparql.setReturnFormat(JSON)
#   results = sparql.query().convert()
#
# GraphDB/Stardog 는 OWL 추론을 DB 차원에서 자동 수행(메모리 추론기 불필요).
# Ontop 은 PostgreSQL 위에서 SPARQL→SQL 재작성(OWL QL).
""")

---
## 📝 정리

1. **온톨로지 = 사실(ABox) + 스키마·규칙(TBox)** → 명시 안 한 사실을 **추론**
2. **그래프 RAG(30) vs 온톨로지 RAG(31)** 의 차이는 DB가 아니라 **추론 계층**
   - 멀티홉 탐색은 둘 다 가능(겹침) / **추론·제약 검사는 온톨로지 RAG 만**
3. `rdflib` 로 **RDF + RDFS/OWL** 구축, **SPARQL** 질의, **전방향 추론**
   - subClassOf(계층), SymmetricProperty(대칭), inverseOf(역관계)
4. **제약 검사**(1인 1회장)로 일관성 검증
5. **온톨로지 RAG + LLM**: *추론으로 보강된* 그래프를 검색해 LLM 답변
6. 실서비스 DB: **Fuseki / GraphDB / Ontop** (RDF/OWL), *Neo4j·AGE 아님*

### 다음 단계
- owlrl 등 완전한 OWL 추론기 적용
- Fuseki/GraphDB 에 적재 후 SPARQL 엔드포인트 서비스화
- 도메인 온톨로지 확장 (법률·금융·의료)